# Faruq-v3 AF2R — Kaggle one-click screening

Tambahkan satu **private Kaggle Dataset** yang berisi tiga file berikut:

1. `faruq-development-v3-grouped.tar.bin` (TAR dengan ekstensi biner agar Kaggle tidak mengubah anotasi)
2. checkpoint AF2 `best.pt` (boleh dinamai `AF2_seed42_best.pt`)
3. `lfdet_afab_seed42_screening.json`

Aktifkan **GPU T4 x2** (device 0) dan Internet; jangan pilih P100 karena image PyTorch Kaggle terbaru tidak menyediakan kernel Pascal/sm_60. Untuk eksekusi tanpa menjaga browser, pilih **Save Version → Save & Run All**. Notebook menjalankan static audit, AF2R0, AF2R1, lalu decision. Test tidak tersedia dan tidak dibaca.


In [ ]:
import importlib,importlib.metadata,json,os,shutil,subprocess,sys,tarfile,time,torch
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
def unique(name):
    matches=sorted(path for path in INPUT.rglob(name) if path.is_file())
    if len(matches)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {matches}')
    return matches[0]
af2_named=sorted(path for path in INPUT.rglob('AF2_seed42_best.pt') if path.is_file())
AF2=af2_named[0] if len(af2_named)==1 else unique('best.pt')
REFERENCE=unique('lfdet_afab_seed42_screening.json')
REPO=WORK/'coffee-bean-detection'; BRANCH='agent/af2-adaptive-residual-gate'
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
TORCH_VERSION_BEFORE=importlib.metadata.version('torch')
subprocess.run([sys.executable,'-m','pip','install','-q','--disable-pip-version-check','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
TORCH_VERSION_AFTER=importlib.metadata.version('torch')
if TORCH_VERSION_AFTER!=TORCH_VERSION_BEFORE: raise RuntimeError(f'Instalasi mengubah Torch {TORCH_VERSION_BEFORE} -> {TORCH_VERSION_AFTER}; restart session.')
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import ultralytics
if ultralytics.__version__!='8.4.96': raise RuntimeError(f'Versi Ultralytics salah: {ultralytics.__version__}')
from coffee_detector.experiments.prepare_faruq_v3_kaggle import prepare_faruq_v3_kaggle_input
DATA,INPUT_CONTRACT=prepare_faruq_v3_kaggle_input(INPUT,WORK)
from ultralytics.data.utils import check_det_dataset
resolved_dataset=check_det_dataset(str(DATA/'data.yaml'))
if Path(resolved_dataset['train']).resolve()!=(DATA/'train/images').resolve(): raise RuntimeError(f'Ultralytics train path salah: {resolved_dataset["train"]}')
if Path(resolved_dataset['val']).resolve()!=(DATA/'val/images').resolve(): raise RuntimeError(f'Ultralytics val path salah: {resolved_dataset["val"]}')
assert torch.cuda.is_available(),'Aktifkan GPU T4 x2 pada Notebook options.'
GPU_NAME=torch.cuda.get_device_name(0); GPU_CAPABILITY=torch.cuda.get_device_capability(0)
if GPU_CAPABILITY[0] < 7: raise RuntimeError(f'GPU {GPU_NAME} tidak kompatibel; pilih GPU T4 x2 dan restart session.')
try: _cuda_probe=torch.ones(1,device='cuda:0').sum().item()
except Exception as exc: raise RuntimeError(f'Kernel CUDA gagal pada {GPU_NAME}: {exc}') from exc
OUTPUT=WORK/'faruq-v3-af2-adaptive-residual-v1'
OUTPUT.mkdir(parents=True,exist_ok=True)
(OUTPUT/'input_contract.json').write_text(json.dumps(INPUT_CONTRACT,indent=2)+'\n',encoding='utf-8')
print('INPUT CONTRACT:',INPUT_CONTRACT); print('GPU:',GPU_NAME); print('DATA:',DATA); print('AF2:',AF2); print('OUTPUT:',OUTPUT)


In [ ]:
from coffee_detector.af2r.audit import run_af2r_static_audit
STATIC=OUTPUT/'static_audit.json'
audit=run_af2r_static_audit(AF2,STATIC,device='cpu')
print('STATIC PARAMETERS:',{'source':audit['source_parameters'],'candidate':audit['candidate_parameters'],'added':audit['added_parameters']})
print('STATIC GATES:',audit['gates']); print('STATIC DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: static audit gagal; training tidak dijalankan.'


In [ ]:
def run_arm(arm):
    result_path=OUTPUT/'val_reports'/f'{arm}_seed42_result.json'
    if result_path.is_file():
        print('REUSE SELESAI:',arm); return json.loads(result_path.read_text())
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2r_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    log=OUTPUT/f'{arm}_seed42_run.log'; log.parent.mkdir(parents=True,exist_ok=True)
    print('START:',arm,'| log=',log,flush=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=REPO,stdout=handle,stderr=subprocess.STDOUT)
        while process.poll() is None:
            try: process.wait(timeout=300)
            except subprocess.TimeoutExpired:
                csv=OUTPUT/arm/f'{arm}_seed42/results.csv'; epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
                print(f'{arm}: {epochs}/30 epoch tercatat',flush=True)
    if process.returncode:
        tail='\n'.join(log.read_text(errors='replace').splitlines()[-150:])
        print(tail,flush=True)
        raise RuntimeError(f'{arm} gagal: {process.returncode}\n--- LOG TERAKHIR ---\n{tail}')
    print('SELESAI:',arm); return json.loads(result_path.read_text())
results={arm:run_arm(arm) for arm in ('AF2R0','AF2R1')}
print({arm:{k:v for k,v in result['metrics'].items() if k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')} for arm,result in results.items()})


In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2r_decision import run_faruq_v3_af2r_decision
decision=run_faruq_v3_af2r_decision(OUTPUT,REFERENCE,seed=42)
import pandas as pd
from IPython.display import display
rows=[{'model':model,**metrics} for model,metrics in decision['values'].items()]
display(pd.DataFrame(rows).style.format({c:'{:.2%}' for c in ['macro_map50_95','bottom3_class_map50_95','worst_class_map50_95']}))
print('AF2R1 vs AF2R0:',decision['af2r1_minus_af2r0']); print('AF2R1 vs AF2:',decision['af2r1_minus_fixed_af2']); print('CRITERIA:',decision['criteria']); print('DECISION:',decision['decision']); print('NEXT:',decision['next']); print('TEST:',decision['test_opened'])


In [ ]:
archive_path=shutil.make_archive(str(WORK/'faruq-v3-af2r-screening-output'),'zip',root_dir=OUTPUT)
print('OUTPUT FOLDER:',OUTPUT); print('DOWNLOAD ZIP:',archive_path)
print('Simpan versi notebook ini. Seluruh checkpoint, log, report, dan decision menjadi Kaggle Output.')
